# BPTT & the Vanishing Gradient

**Companion lesson:** https://ml-viz.vercel.app/courses/rnns/02-bptt-and-vanishing-gradient

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## 1. A vanilla RNN we can differentiate

To *see* the vanishing gradient we implement BPTT by hand. Start with the forward pass, caching every hidden state.

In [ ]:
def rnn_forward(xs, Wxh, Whh, Why, bh, by):
    h = np.zeros((Whh.shape[0], 1))
    hs = [h]
    for x in xs:
        h = np.tanh(Wxh @ x + Whh @ h + bh)
        hs.append(h)
    y = Why @ hs[-1] + by      # single output at the final step
    return y, hs

n_in, nh = 2, 4
Wxh = np.random.randn(nh, n_in) * 0.5
Whh = np.random.randn(nh, nh) * 0.5
Why = np.random.randn(1, nh) * 0.5
bh, by = np.zeros((nh, 1)), np.zeros((1, 1))
xs = [np.random.randn(n_in, 1) for _ in range(20)]
y, hs = rnn_forward(xs, Wxh, Whh, Why, bh, by)
print('output after 20 steps:', round(y.item(), 4))

## 2. BPTT from scratch

Backprop the loss at the **final** step back through every time step. The key quantity is $\partial \mathcal{L}/\partial h_t$ — how much the loss depends on the hidden state $t$ steps in the past. We record its norm at every step.

The recurrence is $h_t = \tanh(a_t)$ with pre-activation $a_t = W_{hh}h_{t-1} + W_{xh}x_t + b$. Differentiating one step gives the Jacobian

$$\frac{\partial h_t}{\partial h_{t-1}} = \mathrm{diag}\big(\tanh'(a_t)\big)\,W_{hh}, \qquad \tanh'(a)=1-\tanh^2(a)=1-h_t^2.$$

In backprop we use its transpose, so per step:

$$dh_{raw} = (1-h_t^2)\odot dh_t,\qquad dh_{t-1} = W_{hh}^\top\, dh_{raw}.$$

The gradient is multiplied by $W_{hh}^\top$ (and a $\tanh'$ diagonal) **once per step it travels back** — that repeated multiplication is the entire vanishing/exploding story.

In [ ]:
def bptt(xs, hs, Wxh, Whh, Why, target):
    T = len(xs)
    y = Why @ hs[-1]
    dy = (y - target)                       # d(0.5*(y-t)^2)/dy
    dWhy = dy @ hs[-1].T
    dh = Why.T @ dy                          # dL/dh_T
    dWxh = np.zeros_like(Wxh); dWhh = np.zeros_like(Whh)
    dh_norms = []
    for t in reversed(range(T)):
        dh_norms.append(np.linalg.norm(dh))  # ||dL/dh_t||
        draw = (1 - hs[t+1]**2) * dh
        dWxh += draw @ xs[t].T
        dWhh += draw @ hs[t].T
        dh = Whh.T @ draw                    # propagate one step back
    return dWxh, dWhh, dWhy, dh_norms[::-1]  # norms ordered t=1..T

_, _, _, norms = bptt(xs, hs, Wxh, Whh, Why, target=np.array([[1.0]]))
print('||dL/dh_t|| for t=1..20:')
print(np.round(norms, 6))

## 2b. The product-of-Jacobians bound, worked by hand

Before trusting the full RNN, pin down the math with a tiny **pure-stdlib** model (no numpy, so the output is deterministic and hand-checkable). Taking norms of the chain rule and using submultiplicativity $\|AB\|\le\|A\|\,\|B\|$:

$$\left\|\frac{\partial \mathcal{L}_T}{\partial h_k}\right\| \le \left\|\frac{\partial \mathcal{L}_T}{\partial h_T}\right\|\,\big(\|W_{hh}\|\cdot\gamma\big)^{T-k},\qquad \gamma=\max_z\tanh'(z)=1.$$

So the per-step gain is the spectral radius of $W_{hh}$ times $\gamma\le 1$. We:

1. multiply the scalar per-step factor explicitly $n$ times and check it equals $(\text{radius})^n$;
2. raise diagonal $2\times2$ matrices $\mathrm{diag}(0.5,0.3)$ and $\mathrm{diag}(1.5,1.2)$ to the $n$-th power and confirm the norm tracks $0.5^n$ (vanish) and $1.5^n$ (explode);
3. show a saturated $\tanh'\approx0.4$ only makes vanishing worse.

In [ ]:
import math  # pure stdlib: deterministic, hand-checkable, no numpy needed

def tanh_prime(z):
    return 1.0 - math.tanh(z) ** 2

# gamma = max tanh'(z) = tanh'(0) = 1; saturation only shrinks it
print("max tanh'(z) = tanh'(0) =", round(tanh_prime(0.0), 6), "(gamma <= 1 always)")
print("saturated tanh'(2)      =", round(tanh_prime(2.0), 6), "-> shrinks the gain further")
print()

# (1) scalar surrogate: per-step factor (w*g) multiplied EXPLICITLY n times
def grad_factor_product(w, g, n):
    factor = 1.0
    for _ in range(n):
        factor *= (w * g)
    return factor

print("gap | radius=0.5 (vanish)   | radius=1.5 (explode)")
for n in [1, 5, 10, 20, 50]:
    dec, exp = grad_factor_product(0.5, 1.0, n), grad_factor_product(1.5, 1.0, n)
    assert abs(dec - 0.5 ** n) <= 1e-12 * max(1.0, 0.5 ** n)   # product == power
    assert abs(exp - 1.5 ** n) <= 1e-9  * max(1.0, 1.5 ** n)
    print(f" {n:>3}| {dec:.6e}        | {exp:.6e}")
print()

# (2) the real product-of-Jacobians: diagonal 2x2 matrices raised to the n-th power
def matmul(A, B):
    return [[sum(A[i][k] * B[k][j] for k in range(len(B))) for j in range(len(B[0]))]
            for i in range(len(A))]

def matpow(M, n):
    R = [[1.0, 0.0], [0.0, 1.0]]            # identity
    for _ in range(n):
        R = matmul(R, M)
    return R

def fro_norm(M):
    return math.sqrt(sum(v * v for row in M for v in row))

W_vanish = [[0.5, 0.0], [0.0, 0.3]]         # spectral radius 0.5
W_explode = [[1.5, 0.0], [0.0, 1.2]]        # spectral radius 1.5
print("gap | ||W_vanish^n||_F (~0.5^n) | ||W_explode^n||_F (~1.5^n)")
for n in [1, 5, 10, 20]:
    nv, ne = fro_norm(matpow(W_vanish, n)), fro_norm(matpow(W_explode, n))
    assert abs(nv - math.sqrt(0.5 ** (2 * n) + 0.3 ** (2 * n))) <= 1e-9
    print(f" {n:>3}| {nv:.6e}              | {ne:.6e}")
# the norm is dominated by the largest eigenvalue^n
assert abs(fro_norm(matpow(W_vanish, 20)) - 0.5 ** 20) <= 1e-6
print()

# (3) tanh' < 1 makes vanishing strictly worse: gain 0.5 * 0.4 = 0.2
print("saturated tanh' ~ 0.4 -> per-step gain 0.5*0.4 = 0.2:")
for n in [5, 10, 20]:
    v = grad_factor_product(0.5, 0.4, n)
    assert abs(v - 0.2 ** n) <= 1e-12 * max(1.0, 0.2 ** n)
    print(f"   gap={n:>2}: ||grad|| ~ {v:.6e}")
print()
print("VERIFY explicit product == closed-form power, and ||diag^n|| ~ radius^n: PASS")

## 3. Verify BPTT with a numerical gradient check

A correct backprop matches finite differences. This proves the implementation.

In [ ]:
def loss_fn(Wxh, Whh, Why):
    y, _ = rnn_forward(xs, Wxh, Whh, Why, bh, by)
    return 0.5 * (y.item() - 1.0)**2

_, dWhh, _, _ = bptt(xs, hs, Wxh, Whh, Why, target=np.array([[1.0]]))
eps = 1e-5; num = np.zeros_like(Whh)
for i in range(Whh.shape[0]):
    for j in range(Whh.shape[1]):
        Whh[i,j]+=eps; lp=loss_fn(Wxh,Whh,Why); Whh[i,j]-=2*eps; lm=loss_fn(Wxh,Whh,Why); Whh[i,j]+=eps
        num[i,j]=(lp-lm)/(2*eps)
rel = np.linalg.norm(dWhh-num)/(np.linalg.norm(dWhh)+np.linalg.norm(num)+1e-12)
print('relative error analytic vs numeric:', f'{rel:.2e}  (<1e-6 = correct)')

## 4. Why it vanishes — gradient norm vs distance

Run BPTT for a 50-step sequence with $W_{hh}$ scaled to different spectral radii. $\|\partial\mathcal{L}/\partial h_t\|$ decays/grows **exponentially** with how far back $t$ is — vanishing when the radius is below 1, exploding when above.

In [ ]:
def gradient_flow(radius, T=50, nh=8):
    W = np.random.randn(nh, nh)
    W *= radius / max(np.abs(np.linalg.eigvals(W)))   # set spectral radius
    Wx = np.random.randn(nh, 1) * 0.3
    Wy = np.random.randn(1, nh) * 0.3
    b = np.zeros((nh, 1))
    xs = [np.random.randn(1, 1) for _ in range(T)]
    _, hs = rnn_forward(xs, Wx, W, Wy, b, np.zeros((1,1)))
    _, _, _, norms = bptt(xs, hs, Wx, W, Wy, target=np.array([[1.0]]))
    return np.array(norms)

T = 50; steps_back = np.arange(T)[::-1]
for radius, col in [(0.5,'#38bdf8'), (0.9,'#14b8a6'), (1.1,'#f59e0b'), (1.3,'#f43f5e')]:
    norms = gradient_flow(radius, T)
    plt.semilogy(steps_back, norms, label=f'spectral radius {radius}', color=col)
plt.xlabel('steps back in time  (T - t)'); plt.ylabel('||dL/dh_t||  (log scale)')
plt.title('Vanishing vs exploding gradients in a real RNN'); plt.legend(); plt.show()

## 5. The consequence: long-range tasks fail

Train the *same* vanilla RNN to recall the **first** input bit after a delay of $T$ steps. It learns short delays and fails at long ones — exactly because the gradient from the final loss vanishes before reaching step 1.

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))

def recall_first(T, epochs=600, lr=0.3, nh=16, seed=0):
    rng=np.random.RandomState(seed)
    Wxh=rng.randn(nh,1)*0.3; Whh=rng.randn(nh,nh)*0.3
    Why=rng.randn(1,nh)*0.3; bh=np.zeros((nh,1)); by=np.zeros((1,1))
    for ep in range(epochs):
        X=rng.randint(0,2,size=(16,T)); tgt=X[:,0]
        gWxh=gWhh=gWhy=gbh=gby=0
        for b in range(16):
            hs=[np.zeros((nh,1))]; xs=[]
            for t in range(T):
                x=np.array([[X[b,t]]],float); xs.append(x)
                hs.append(np.tanh(Wxh@x+Whh@hs[-1]+bh))
            p=sigmoid(Why@hs[-1]+by); dy=p-tgt[b]
            gWhy=gWhy+dy@hs[-1].T; gby=gby+dy; dh=Why.T@dy
            for t in reversed(range(T)):
                draw=(1-hs[t+1]**2)*dh
                gWxh=gWxh+draw@xs[t].T; gWhh=gWhh+draw@hs[t].T; gbh=gbh+draw
                dh=Whh.T@draw
        # NOTE: no gradient clipping here, so the vanishing gradient is free to bite
        for p,g in [(Wxh,gWxh),(Whh,gWhh),(Why,gWhy),(bh,gbh),(by,gby)]:
            p-=lr*g/16
    # evaluate
    Xt=rng.randint(0,2,size=(300,T)); correct=0
    for b in range(300):
        h=np.zeros((nh,1))
        for t in range(T):
            h=np.tanh(Wxh@np.array([[Xt[b,t]]],float)+Whh@h+bh)
        pred=(sigmoid(Why@h+by)>0.5).item()
        correct += int(pred==bool(Xt[b,0]))
    return correct/300

for T in [5, 15, 30, 50]:
    acc = recall_first(T)
    print(f'delay T={T:>2}: recall accuracy = {acc:.2f}')
print('\nShort delays learn (~1.0); long delays collapse to chance (~0.5) — the gradient vanished before reaching step 1.')

## Key takeaways

- BPTT multiplies the gradient by $W_{hh}^\top$ once per step it travels back.
- $\|\partial\mathcal{L}/\partial h_t\|$ therefore decays or grows **exponentially** with distance.
- We verified the hand-written BPTT against a numerical gradient check.
- The vanishing gradient is not abstract: the same RNN provably fails to recall a bit across a long delay.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The gradient factor

For a scalar linear RNN, the gradient flowing back $T$ steps is multiplied by $w_{rec}$ once per step:

$$\left| \frac{\partial h_T}{\partial h_0} \right| = |w_{rec}|^T$$

Implement the factor and check the three regimes from section 2b: $|w| < 1$ vanishes geometrically, $|w| > 1$ explodes, and only $|w| = 1$ carries gradients undamped.

In [ ]:
def gradient_factor(w_rec, T):
    """|dh_T / dh_0| for a scalar linear RNN."""
    # TODO(you): |w_rec| to the power T
    return ...

In [ ]:
# Checks — run me
assert abs(gradient_factor(0.5, 10) - 0.5 ** 10) < 1e-15, "|w|^T"
assert gradient_factor(0.5, 10) < 1e-3, "w = 0.5: ten steps shrink the gradient 1000x"
assert gradient_factor(1.2, 40) > 1e3, "w > 1: the same product explodes"
assert gradient_factor(1.0, 1000) == 1.0, "only |w| = 1 carries gradients undamped"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def gradient_factor(w_rec, T):
    return abs(w_rec) ** T
```

</details>

### Exercise 2 — The effective horizon

Turn the decay into a memory length: the **effective horizon** is the first $T$ where the gradient factor drops below a threshold — beyond it, the network effectively can't learn the dependency. The checks quantify how hopeless it is: even $w = 0.99$ forgets within ~700 steps.

In [ ]:
def effective_horizon(w_rec, threshold=1e-3):
    """Smallest T with gradient_factor(w_rec, T) < threshold."""
    T = 1

    # TODO(you): increment T until the factor drops below the threshold
    while ...:
        T += 1

    return T

In [ ]:
# Checks — run me
assert effective_horizon(0.5) == 10, "0.5^10 ~ 9.8e-4 is the first below 1e-3"
assert effective_horizon(0.9) > effective_horizon(0.5), "w closer to 1 -> longer memory"
assert effective_horizon(0.99, 1e-3) > 600, "even w = 0.99 forgets within ~700 steps"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def effective_horizon(w_rec, threshold=1e-3):
    T = 1
    while gradient_factor(w_rec, T) >= threshold:
        T += 1
    return T
```

</details>